# Biblical Qwen3.5 4B DPO Training with Unsloth (4-bit QLoRA)

**Phase 2:** Direct Preference Optimization continuing the Qwen3.5 4B SFT LoRA. DPO sharpens persona
behavior by contrasting preferred persona-voiced answers with generic, shallow, or out-of-voice ones.

**Base Model:** `unsloth/Qwen3.5-4B` — bf16 weights (`Qwen3_5ForConditionalGeneration`, `model_type`
`qwen3_5`), quantized to bnb NF4 on the fly. Multimodal hybrid: 24 gated-delta `linear_attn` + 8
`full_attention` language layers, plus a 24-block vision tower and a 1-layer MTP head.

**Starting Adapter:** `output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters`, produced by
`biblical_qwen3_5_4b_sft_unsloth_4bit.ipynb`. This notebook resolves that path from
`SFT_MODEL_NAME_BASE`, which must match the SFT notebook's `MODEL_NAME_BASE` exactly, and refuses to
start unless that run wrote a `complete.json` sentinel.

**Dataset:** `biblical_personas_v2_dpo.jsonl` — **3,564 preference pairs** across 3 rejection sources
(`shallow_platitude` 1200, `scripture_fabrication` 1200, `voice_drift` 1164) and 15 personas, evenly
distributed (~238 each). Measured through this tokenizer: prompt max **491** tokens, prompt + longer
response max **1506**. `MAX_SEQ_LENGTH = 2048` / `MAX_PROMPT_LENGTH = 1024` drops **nothing**.

**Training Hardware:** NVIDIA DGX Spark (GB10, sm_120, 128 GB unified memory).

**LoRA scope is inherited, not established here.** This notebook does **not** call `get_peft_model`.
It attaches the SFT adapter with `is_trainable=True` and continues training it, so the output stays a
single LoRA against the base. A leaked SFT adapter stays leaked through DPO and into everything
downstream — the model cell asserts the scope *before* the expensive reference-logprob precompute.

---

### Three things verified against the installed stack, not carried over from the 27B notebooks

The 27B DPO notebooks in this repo were written against an older TRL. Each of these was checked in
`trl 0.24.0` / `transformers 5.16.0.dev0` source in this container:

1. **`import unsloth` before `trl` is load-bearing here specifically.** Under `transformers 5.x`,
   `_is_package_available(name)` always returns a `(bool, version)` **tuple**, and `trl 0.24.0`
   assigns it straight into `_llm_blender_available` / `_mergekit_available`
   (`trl/import_utils.py:30-41`). A non-empty tuple is truthy, so every optional-dependency guard
   in TRL fires no matter what is installed: a bare `from trl import DPOTrainer` tries to import
   `llm_blender` **and** `mergekit`, and dies on whichever is absent. Unsloth normalizes
   `_is_package_available` (`unsloth/models/_utils.py:2181`) before `trl` is imported, which is
   what makes those guards behave. The trainer cell asserts this rather than letting it surface as
   a confusing traceback. The flip side is that `mergekit` must be *absent* — see Section 2.
2. **The `model.config.model_type` swap is load-bearing, and post-hoc `trainer.is_vision_model = False`
   is too late.** `DPOTrainer.__init__` sets
   `self.is_vision_model = model.config.model_type in MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES`
   (dpo_trainer.py:359) and then picks `tokenize_row` vs `process_row` from it *during* `__init__`
   (line 667). `qwen3_5` **is** in that mapping; `qwen3_5_text` is not. Assigning the attribute after
   construction changes nothing — the dataset is already tokenized by then.
3. **Unwrapping the Processor is necessary but not sufficient.** It sets TRL's `_is_vlm = False`
   (line 322-329) and fixes `model_input_names`, but `is_vision_model` is read from the *model config*,
   not the processing class. Both fixes are applied, and the trainer cell then asserts the dataset
   actually came out of the text path.

The persistent reference-logprob cache **is** still valid on this TRL: `DataCollatorForPreference`
honors `ref_chosen_logps` / `ref_rejected_logps` when present (dpo_trainer.py:159-179), and
`_set_signature_columns_if_needed` keeps both columns from being pruned (line 807-814).

**Chat template / thinking mode:** prompt formatting always uses `enable_thinking=False`, matching
what SFT trained on. `ENABLE_THINKING` controls only the eval cells.

## 1. Configuration

All paths, hyperparameters, and run-shaping values for this notebook.

In [1]:
import os
from pathlib import Path

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
# Normally run inside the `unsloth-notebook` container, which bind-mounts
# /home/spark/projects/training -> /workspace/training. The fallbacks let the same notebook run
# on the host without edits.
if os.path.exists("/workspace/training/biblical"):
    PROJECT_ROOT = Path("/workspace/training/biblical")
elif os.path.exists("/workspace/biblical"):
    PROJECT_ROOT = Path("/workspace/biblical")
else:
    PROJECT_ROOT = Path("/home/spark/projects/training/biblical")

OUTPUT_ROOT = PROJECT_ROOT / "output"

# =========================== SHARED HUGGING FACE CACHE ===========================
# Must be set BEFORE unsloth/transformers are imported.
for _cache_dir in ("/root/.cache/huggingface", "/home/spark/.cache/huggingface"):
    if os.path.isdir(_cache_dir):
        os.environ["HF_HUB_CACHE"] = _cache_dir
        break

# =========================== ALLOCATOR: DELIBERATELY UNSET ===========================
# Do NOT set PYTORCH_CUDA_ALLOC_CONF here, and in particular do not set max_split_size_mb.
#
# docs/dgx_spark_gb10_quirks.md recommends
#     "garbage_collection_threshold:0.5,max_split_size_mb:256"
# and states the confirmed-working Biblical Qwen3.8 27B run sets it. That statement is wrong:
# neither biblical_qwen3_8_27b_sft_unsloth_4bit.ipynb nor stoic_qwen38_27b_sft.ipynb contains
# PYTORCH_CUDA_ALLOC_CONF anywhere, and it is not set in the container environment either. The
# runs that actually completed on this machine set nothing.
#
# Setting it here caused a reproducible OOM. max_split_size_mb:256 tells the caching allocator
# never to split a block larger than 256 MB. With a 248,320-token vocabulary the logits tensor is
# multiple GB, and because packing=False every batch pads to its own longest member, that tensor
# is a DIFFERENT SIZE almost every step. So a cached 5.8 GB block can never be carved down to
# serve a 5.3 GB request - the allocator must cudaMalloc a fresh one while the old one stays
# reserved. Each new batch shape permanently adds another multi-GB block.
#
# The signature is unmistakable once you look for it: peak ALLOCATION stayed near 20 GB while
# RESERVED climbed 24.9 -> 38.1 GB in eight steps. Live memory was never the problem; the cache
# was. On GB10 host and device share one 128 GB pool, so that climb consumes the machine.
#
# Also deliberately NO torch.cuda.set_per_process_memory_fraction(): on unified memory a
# fractional cap protects nothing and adds a ceiling the bf16 GGUF merge would have to fit under.
# If fragmentation ever does need addressing, "expandable_segments:True" is the setting designed
# for varying allocation sizes - max_split_size_mb is the opposite of what this workload wants.

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM = "unsloth/Qwen3.5-4B"
MODEL_NAME_BASE = "biblical_qwen3_5_4b_unsloth_4bit_dpo"

# MUST match MODEL_NAME_BASE in biblical_qwen3_5_4b_sft_unsloth_4bit.ipynb.
# This is the SFT -> DPO contract. Change it in both places or not at all.
SFT_MODEL_NAME_BASE = "biblical_qwen3_5_4b_unsloth_4bit"
SFT_LORA_PATH = OUTPUT_ROOT / SFT_MODEL_NAME_BASE / "lora_adapters"

# =========================== THINKING MODE ===========================
# Prompt formatting for DPO ALWAYS uses enable_thinking=False (see the formatting cell), matching
# how the SFT training data was built. On this checkpoint's template that equals omitting the
# kwarg - but the SFT notebook does not rely on that and neither does this one.
# This flag controls only the eval cells, which must match training to be a meaningful test.
ENABLE_THINKING = False

# =========================== INPUT DATA ===========================
# Preference pairs from the Biblical v2 DPO datagen notebook.
# Format: {chosen: [system, user, assistant], rejected: [system, user, assistant], source, persona}
DPO_DATA_FILE = PROJECT_ROOT / "data/training-data/biblical_persona_v2/biblical_personas_v2_dpo.jsonl"

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = OUTPUT_ROOT / MODEL_NAME_BASE
TRAIN_DIR = OUTPUT_BASE_DIR / "train"
LORA_OUTPUT_DIR = OUTPUT_BASE_DIR / "lora_adapters"

# =========================== DPO HYPERPARAMETERS ===========================
# Measured over all 3,564 pairs with THIS tokenizer at enable_thinking=False:
#   prompt              p50=406  p90=441  p99=482   max=491
#   chosen response     p50=361  p90=454  p99=560   max=826
#   rejected response   p50=335  p90=618  p99=906   max=1048
#   prompt + max(resp)  p50=821  p90=1031 p99=1310  max=1506
# These pairs are single-turn (system + user + assistant), so they are far shorter than the
# multi-turn SFT conversations. 2048/1024 drops zero pairs with ~36% headroom; the formatting
# cell re-measures and raises if a regenerated dataset ever starts losing rows to the filter.
MAX_SEQ_LENGTH = 2048
MAX_PROMPT_LENGTH = 1024

# DO NOT RAISE BATCH_SIZE - same reason as the SFT notebook, and worse here.
#
# Qwen3.5's vocabulary is 248,320 tokens, so the logits tensor is batch x seq x 248320 and
# dominates memory regardless of how small the 4B's weights are. DPO makes this ~2x worse than
# SFT because concatenated_forward runs chosen AND rejected together, so a per-device batch of N
# is 2N sequences through the LM head.
#
# Measured on this machine, the SFT notebook at per-device batch 4 reached 38.1 GB reserved after
# 8 steps and kept climbing toward the whole 128 GB pool. 1 x 8 is the confirmed 27B DPO recipe.
BATCH_SIZE = 1
GRAD_ACCUM = 8

LEARNING_RATE = 5e-6
TARGET_EPOCHS = 1
DPO_BETA = 0.1
LOSS_TYPE = "sigmoid"
WARMUP_RATIO = 0.1
DPO_MAX_PAIRS = 0  # 0 = use all valid pairs; set e.g. 1200/2400 for capped runs (stratified by source)

# Set True to rebuild the formatted-dataset cache even when its fingerprint matches.
FORCE_REFORMAT = False

# =========================== CHECKPOINTING / MEMORY ===========================
# 3,564 pairs / effective batch 8 = ~446 steps.
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 3
# gc.collect() + torch.cuda.empty_cache() cadence. Every 50 steps, NOT every step - per-step
# cache flushing forces the allocator to re-acquire blocks and measurably slows training.
CLEANUP_STEPS = 50

# =========================== REFERENCE LOGPROB CACHE ===========================
# Shard size trades file count against work lost on a crash. 64 rows/shard over 3,564 rows is
# 56 shards. Batch size 1 is the documented default and keeps peak memory flat.
REF_LOGPROBS_SHARD_SIZE = 64
REF_LOGPROBS_BATCH_SIZE = 1

# =========================== EVAL ===========================
# This repo publishes no generation_config.json for Qwen3.5-4B, so these are the project's
# standard Qwen sampling values, matching the SFT notebook - not values read from the checkpoint.
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.8
GEN_TOP_K = 20

# ============================================================================
print("=" * 60)
print("CONFIGURATION LOADED (Qwen3.5 4B DPO)")
print("=" * 60)
print(f"  Project root:   {PROJECT_ROOT}")
print(f"  HF hub cache:   {os.environ.get('HF_HUB_CACHE', '<default>')}")
print(f"  Base model:     {BASE_LLM}")
print(f"  Model name:     {MODEL_NAME_BASE}")
print(f"  SFT LoRA:       {SFT_LORA_PATH}")
print(f"  DPO data:       {DPO_DATA_FILE}")
print(f"  Output:         {OUTPUT_BASE_DIR}")
print(f"  Training:       batch={BATCH_SIZE} x {GRAD_ACCUM} (effective {BATCH_SIZE*GRAD_ACCUM}), "
      f"lr={LEARNING_RATE}, beta={DPO_BETA}, loss={LOSS_TYPE}")
print(f"  Lengths:        max_seq={MAX_SEQ_LENGTH}, max_prompt={MAX_PROMPT_LENGTH} "
      f"(measured data max: 1506 / 491)")
print(f"  DPO max pairs:  {DPO_MAX_PAIRS or 'ALL (no limit)'}")
print(f"  Checkpoints:    every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Ref cache:      shards of {REF_LOGPROBS_SHARD_SIZE}, batch {REF_LOGPROBS_BATCH_SIZE}")
print(f"  Thinking mode:  {'ON' if ENABLE_THINKING else 'OFF'} (eval cells only)")

# Fail here, not three cells and a model load later.
if not DPO_DATA_FILE.exists():
    raise FileNotFoundError(f"DPO data not found: {DPO_DATA_FILE}")
if not SFT_LORA_PATH.exists():
    raise FileNotFoundError(
        f"SFT LoRA not found at {SFT_LORA_PATH}. Run biblical_qwen3_5_4b_sft_unsloth_4bit.ipynb "
        f"first, and confirm SFT_MODEL_NAME_BASE here matches MODEL_NAME_BASE there."
    )

# The SFT notebook writes complete.json only after the adapter and tokenizer are saved. Refuse to
# spend hours continuing an adapter from a run that did not finish.
import json as _json
_sft_sentinel_path = SFT_LORA_PATH / "complete.json"
if not _sft_sentinel_path.exists():
    raise FileNotFoundError(
        f"No completion sentinel at {_sft_sentinel_path}. The SFT run did not finish (or predates "
        "the sentinel). Re-run the SFT notebook to completion before starting DPO."
    )
_sft_sentinel = _json.loads(_sft_sentinel_path.read_text())
if _sft_sentinel.get("status") != "complete":
    raise RuntimeError(f"SFT sentinel status is {_sft_sentinel.get('status')!r}, expected 'complete'.")
if _sft_sentinel.get("base_model") != BASE_LLM:
    raise RuntimeError(
        f"SFT adapter was trained on {_sft_sentinel.get('base_model')!r} but this notebook loads "
        f"{BASE_LLM!r}. The adapter will not match the base."
    )

print(f"\n  SFT adapter present and complete:")
print(f"    step {_sft_sentinel.get('global_step')}, loss {_sft_sentinel.get('final_loss')}, "
      f"{_sft_sentinel.get('adapted_modules')} adapted modules, "
      f"{len(_sft_sentinel.get('personas', []))} personas")

CONFIGURATION LOADED (Qwen3.5 4B DPO)
  Project root:   /workspace/training/biblical
  HF hub cache:   /root/.cache/huggingface
  Base model:     unsloth/Qwen3.5-4B
  Model name:     biblical_qwen3_5_4b_unsloth_4bit_dpo
  SFT LoRA:       /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters
  DPO data:       /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v2_dpo.jsonl
  Output:         /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo
  Training:       batch=1 x 8 (effective 8), lr=5e-06, beta=0.1, loss=sigmoid
  Lengths:        max_seq=2048, max_prompt=1024 (measured data max: 1506 / 491)
  DPO max pairs:  ALL (no limit)
  Checkpoints:    every 50 steps, keep 3
  Ref cache:      shards of 64, batch 1
  Thinking mode:  OFF (eval cells only)

  SFT adapter present and complete:
    step 544, loss 1.6775256220032186, 128 adapted modules, 26 personas


## 2. Environment Preparation

Idempotent — safe to re-run. Identical to the SFT notebook's, and for the same reasons:

1. Verify the CUDA PyTorch build is intact (no side effects).
2. Core training packages; remove the aarch64 `torchao` build that blocks PEFT's bnb 4-bit
   dispatcher, and remove `mergekit`, whose model definitions no longer build under this
   container's `pydantic 2.13` (see below).
3. `transformers` + `peft` from git main — **only if** the installed `transformers` does not already
   register the `qwen3_5` architecture. It currently does, so this normally skips.
4. Small utility packages.
5. Rebuild `causal_conv1d` from source if its compiled CUDA extension is missing; uninstall it if
   the build fails, so the PyTorch fallback path is taken cleanly.
6. `flash-linear-attention` — Triton kernels for the gated-delta-rule fast path.
7. Import `unsloth` **before** `transformers`/`trl`/`peft`.

Steps 2 and 7 are the two halves of one problem, and both are needed on this stack.

`transformers 5.x` makes `_is_package_available(name)` always return a `(bool, version)` tuple,
while `trl 0.24.0` still assigns it directly to its `_*_available` flags. A non-empty tuple is
truthy, so TRL's optional-dependency guards all fire and a bare `from trl import DPOTrainer`
imports `llm_blender` and `mergekit` whether or not they exist:

```
ModuleNotFoundError: No module named 'llm_blender'        # trl/trainer/judges.py, behind a broken guard
```

Importing `unsloth` first fixes that — it normalizes `_is_package_available` before `trl` loads
(Step 7). But that only covers packages that are *missing*. `mergekit` is installed in this image,
and importing it is what breaks now: `pydantic 2.13` builds a schema for the `execute` callable
that `mergekit/merge_methods/easy_define.py` generates, and dies on its `torch.Tensor` argument.
Unsloth reaches that import on its own, so `import unsloth` fails before any of this notebook's
code runs:

```
import unsloth
  -> unsloth_zoo.temporary_patches.misc.patch_trl_vision_model_mapping
  -> trl.trainer.dpo_trainer -> trl.trainer.callbacks -> trl.mergekit_utils -> mergekit
PydanticSchemaGenerationError: Unable to generate pydantic-core schema for <class 'torch.Tensor'>
```

Step 2 uninstalls `mergekit` (TRL wants it only for `MergeModelCallback`, unused here). The trainer
cell then checks the whole import path explicitly.

**If this cell installs anything, restart the kernel and resume from Cell 1.**

In [2]:
import os, sys, subprocess, importlib, importlib.util

_installed_something = False


def _pip(*args, env_extra=None):
    """Run pip against this kernel's interpreter; print output only on failure."""
    global _installed_something
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args], capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    _installed_something = True
    return True


def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None


print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# --- 1. Verify the CUDA PyTorch build is intact ------------------------------
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "cpu" in torch.__version__:
        print("  CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through, or another container holds it.")
        print("  Check `docker ps` for a running sglang-* / vllm-* server and stop it.")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")
print(f"  torch {torch.__version__} - CUDA {torch.version.cuda} - GPU: {torch.cuda.get_device_name(0)}")

# --- 2. Core training packages ------------------------------------------------
print("  Ensuring core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...")
_pip("install", "-q", "-U", "unsloth", "trl", "accelerate", "datasets", "bitsandbytes")

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires torchao>=0.16.0 OR
# torchao absent. No aarch64 wheel >=0.16 exists on PyPI, so uninstall - peft's torchao dispatcher
# then no-ops and falls through to the bnb 4-bit dispatcher, which is what QLoRA wants anyway.
if importlib.util.find_spec("torchao") is not None:
    print("  Removing torchao (blocks peft's bnb 4-bit dispatcher on aarch64)...")
    _pip("uninstall", "-y", "-q", "torchao")

# mergekit 0.1.4 cannot be imported under this container's pydantic 2.13. Its merge methods are
# declared with pydantic.create_model(), and 2.13 now builds a schema for the generated `execute`
# callable, whose Dict[str, torch.Tensor] argument raises
#     PydanticSchemaGenerationError: Unable to generate pydantic-core schema for torch.Tensor
# That import is not optional for us: unsloth's patch_trl_vision_model_mapping imports
# trl.trainer.dpo_trainer -> trl.trainer.callbacks -> trl.mergekit_utils -> mergekit, so
# `import unsloth` itself dies in Step 7. TRL only needs mergekit for MergeModelCallback, which
# this notebook never uses. Reinstall with `pip install mergekit==0.1.4` if anything else wants it.
if importlib.util.find_spec("mergekit") is not None:
    print("  Removing mergekit (pydantic 2.13 breaks its schema build, killing `import unsloth`)...")
    _pip("uninstall", "-y", "-q", "mergekit")

# --- 3. transformers + peft from git main, ONLY if qwen3_5 is unknown ---------
try:
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    _qwen35_known = "qwen3_5" in CONFIG_MAPPING_NAMES
except Exception:
    _qwen35_known = False

if _qwen35_known:
    print("  transformers already registers `qwen3_5` - skipping git-main reinstall")
else:
    print("  `qwen3_5` not registered - installing transformers + peft from git main...")
    _pip("install", "-q", "-U", "git+https://github.com/huggingface/transformers.git")
    _pip("install", "-q", "-U", "git+https://github.com/huggingface/peft.git")

# --- 4. Small utility packages ------------------------------------------------
for _module, _install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "tqdm":        ["install", "-q", "tqdm"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(_module) is None:
        print(f"  Installing {_install_args[-1]}...")
        _pip(*_install_args)

# --- 5. Fix causal_conv1d -----------------------------------------------------
# The NGC image installs the causal_conv1d Python package WITHOUT its compiled CUDA extension,
# which hard-crashes any import that reaches the FalconH1 model inside transformers or unsloth.
# pip also caches a broken prebuilt aarch64 wheel, so --no-binary forces a source build.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",   # DGX Spark GB10 = sm_120
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing - rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    _ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
               "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if _ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext - uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed - uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# --- 6. flash-linear-attention ------------------------------------------------
if _check_import("fla") is None:
    print("  Installing flash-linear-attention...")
    _pip("install", "-q", "--no-deps", "flash-linear-attention", "fla-core")

_fast_path_ok = False
try:
    from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule
    _fast_path_ok = _causal_ok and chunk_gated_delta_rule is not None
    for _k in list(sys.modules.keys()):
        if _k.startswith("fla."):
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError):
    pass
print(f"  Fast path: {'ENABLED' if _fast_path_ok else 'DISABLED (using torch fallback)'}")

# --- 7. Import unsloth FIRST, then transformers/trl/peft ----------------------
# Purge anything already loaded so the imports pick up whatever was installed above.
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
import peft
import trl

# Prove the DPO import path works in THIS kernel. Importing unsloth first is load-bearing, and the
# reason is mechanical: under transformers 5.x `_is_package_available(name)` always returns a
# (bool, version) TUPLE, and trl 0.24.0 assigns it straight into `_llm_blender_available` /
# `_mergekit_available` (trl/import_utils.py:30-41). A non-empty tuple is truthy, so every
# optional-dependency guard in TRL fires regardless of what is installed, and a bare
# `from trl import DPOTrainer` tries to import llm_blender and mergekit, dying on whichever is
# absent. Unsloth normalizes _is_package_available (unsloth/models/_utils.py:2181) before trl is
# imported, which is what makes those guards behave. Fail loudly here rather than five cells later.
from trl import DPOTrainer as _DPOTrainerProbe
del _DPOTrainerProbe

print()
print(f"  unsloth:       {unsloth.__version__}")
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  trl:           {trl.__version__}")
print(f"  DPOTrainer:    importable")
print(f"  causal_conv1d: {'OK' if _causal_ok else 'FALLBACK (torch path)'}")
print(f"  fla:           {'OK' if _check_import('fla') else 'MISSING'}")
print(f"  torchao:       {'PRESENT (should be absent)' if importlib.util.find_spec('torchao') else 'absent (correct)'}")
print(f"  mergekit:      {'PRESENT (should be absent)' if importlib.util.find_spec('mergekit') else 'absent (correct)'}")
print()
if _installed_something:
    print("Packages were installed/removed. RESTART THE KERNEL, then rerun from Cell 1.")
else:
    print("Environment already satisfied - no installs, no restart needed. Continue to Cell 3.")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 - CUDA 13.0 - GPU: NVIDIA GB10
  Ensuring core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...
  Removing torchao (blocks peft's bnb 4-bit dispatcher on aarch64)...
  transformers already registers `qwen3_5` - skipping git-main reinstall
  causal_conv1d: OK (CUDA extension loaded)
  Fast path: ENABLED
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

  unsloth:       2026.8.22
  transformers:  5.5.0
  peft:          0.20.1.dev0
  trl:           0.24.0
  DPOTrainer:    importable
  causal_conv1d: OK
  fla:           OK
  torchao:       absent (correct)
  mergekit:      absent (correct)

Packages were installed/removed. RESTART THE KERNEL, then rerun from Cell 1.


## 3. Load DPO Dataset

Load the preference pairs produced by the Biblical v2 DPO datagen notebook.

Expected per-line format:
`{"chosen": [system, user, assistant], "rejected": [system, user, assistant], "source": str, "persona": str}`

The persona system prompts are extracted here so they can be saved next to the adapter, exactly as
the SFT notebook does.

In [3]:
import json
from collections import Counter

with open(DPO_DATA_FILE) as f:
    raw_pairs = [json.loads(line) for line in f if line.strip()]

if not raw_pairs:
    raise RuntimeError(f"No rows read from {DPO_DATA_FILE}")

source_counts = Counter(pair.get("source", "unknown") for pair in raw_pairs)
persona_counts = Counter(pair.get("persona", "unknown") for pair in raw_pairs)

persona_system_prompts = {}
for pair in raw_pairs:
    persona = pair.get("persona", "unknown")
    chosen = pair.get("chosen", [])
    if chosen and chosen[0].get("role") == "system":
        persona_system_prompts.setdefault(persona, chosen[0].get("content", ""))

print(f"Loaded {len(raw_pairs):,} DPO pairs from {DPO_DATA_FILE}")
print(f"  Personas:                {len(persona_counts)}")
print(f"  System prompts extracted: {len(persona_system_prompts)}")

print("\nSource distribution (rejection strategy):")
for source, count in source_counts.most_common():
    print(f"  {source:<28} {count:>6}  ({100 * count / len(raw_pairs):.1f}%)")

print("\nPersona distribution:")
for persona, count in persona_counts.most_common():
    print(f"  {persona:<28} {count:>6}")

# A single rejection source dominating means DPO optimizes against one failure mode only.
_dominant = source_counts.most_common(1)[0]
if _dominant[1] / len(raw_pairs) > 0.6:
    print(f"\n  WARNING: '{_dominant[0]}' is {100 * _dominant[1] / len(raw_pairs):.0f}% of all pairs. "
          "DPO will mostly learn to avoid that one failure mode.")

Loaded 3,564 DPO pairs from /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v2_dpo.jsonl
  Personas:                26
  System prompts extracted: 26

Source distribution (rejection strategy):
  shallow_platitude              1200  (33.7%)
  scripture_fabrication          1200  (33.7%)
  voice_drift                    1164  (32.7%)

Persona distribution:
  peter                           240
  solomon                         239
  david                           239
  paul                            238
  moses                           238
  daniel                          238
  zechariah                       238
  ezekiel                         237
  job                             237
  jeremiah                        236
  hosea                           235
  isaiah                          234
  amos                            174
  joshua                           87
  micah                            71
  apostle_john                     

## 4. Load Model + SFT LoRA (4-bit)

Load the base in 4-bit, then attach the saved SFT adapter with `PeftModel.from_pretrained(...,
is_trainable=True)`. This is the pattern that produced the working Qwen3.8 27B DPO adapter on this
machine: it keeps the adapter trainable and bypasses Unsloth's inference-oriented adapter auto-reload
path, so DPO continues the *same* LoRA rather than stacking a second one.

**`use_gradient_checkpointing=True` is passed explicitly, and it matters more here than in SFT.**
`FastLanguageModel.from_pretrained` defaults this to `"unsloth"` (unsloth/models/vision.py:898).
That value routes through `patch_unsloth_smart_gradient_checkpointing`, which swaps
`torch.utils.checkpoint.CheckpointFunction`, `torch.utils.checkpoint.checkpoint`, and
`transformers.modeling_utils.checkpoint` for offloading versions that stage saved activations in
**pinned host buffers** (`unsloth_zoo/gradient_checkpointing.py:810` —
`torch.empty(..., device="cpu", pin_memory=True)`) which only ever grow.

On a discrete GPU that is a good trade. On GB10 host and device are the **same 128 GB pool**, so
the copy frees nothing while ratcheting unreclaimable page-locked memory upward for hours.

The SFT notebook neutralizes this by passing `use_gradient_checkpointing=True` to
`get_peft_model`, which calls `unpatch_unsloth_smart_gradient_checkpointing()`. **A DPO notebook
never calls `get_peft_model`** — it continues the existing adapter — so nothing unpatches it and
the offload path stays live for the whole run. Verified by observation: a probe run of this
notebook without the flag printed
`Unsloth: Will smartly offload gradients to save VRAM!` at step 1. The cell below passes the flag
and then asserts the patch is actually gone, checking the swapped functions by name.

The inherited-scope assertion runs here too, immediately after the adapter attaches and before any
expensive work.

In [4]:
import os
# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and the torch.compile path.
# Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
from peft import PeftModel
import torch

print(f"Loading base: {BASE_LLM}")
model, tokenizer = FastLanguageModel.from_pretrained(
    str(BASE_LLM),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    # Explicitly True, NOT the "unsloth" default (vision.py:898). See this cell's markdown:
    # "unsloth" stages saved activations in pinned host buffers that only grow, and on GB10 host
    # and device share one pool so that frees nothing. Passing True calls
    # unpatch_unsloth_smart_gradient_checkpointing(). Unlike the SFT notebook, this is the ONLY
    # place it can be set - DPO never calls get_peft_model.
    use_gradient_checkpointing=True,
)

# Assert the offloading patch is actually gone. patch_unsloth_smart_gradient_checkpointing swaps
# these three by name (unsloth_zoo/gradient_checkpointing.py:1116-1134); the unpatch restores all
# three. Checking the names is a direct read of which implementation will run, rather than trusting
# that the kwarg was threaded through.
import torch.utils.checkpoint as _tuc
import transformers.modeling_utils as _tmu
_gc_patched = [
    name for name, fn in (
        ("torch.utils.checkpoint.CheckpointFunction", _tuc.CheckpointFunction),
        ("torch.utils.checkpoint.checkpoint", _tuc.checkpoint),
        ("transformers.modeling_utils.checkpoint", _tmu.checkpoint),
    )
    if getattr(fn, "__name__", "") in ("UnslothCheckpointFunction", "unsloth_checkpoint")
]
if _gc_patched:
    raise RuntimeError(
        f"Unsloth's offloading gradient checkpointing is still patched in: {_gc_patched}. "
        "It stages activations in pinned host memory that only grows - on GB10 that frees nothing "
        "and ratchets unreclaimable memory upward for the whole run. Restart the kernel and load "
        "with use_gradient_checkpointing=True."
    )
print("  Gradient checkpointing: standard recompute (unsloth host-offload patch not active)")

# ---- Multimodal base: unwrap the Processor -------------------------------------
# Qwen3.5-4B is Qwen3_5ForConditionalGeneration, so Unsloth returns a multimodal Processor.
# Unwrapping sets TRL's internal `_is_vlm = False` (dpo_trainer.py:322-329) and makes
# model_input_names[0] be "input_ids" rather than "pixel_values".
#
# This is NECESSARY BUT NOT SUFFICIENT: DPOTrainer also computes
#   self.is_vision_model = model.config.model_type in MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES
# from the MODEL CONFIG, not the processing class (line 359), and picks tokenize_row vs
# process_row from it during __init__ (line 667). The trainer cell handles that half.
# `processor` is kept so the adapter directory is saved with the same files the SFT adapter has.
processor = None
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from Processor - text-only DPO mode)")
text_tokenizer = tokenizer   # explicit alias used by the ref-logprob cache cell

# This checkpoint ships pad_token = '<|vision_pad|>' (248055), distinct from eos = '<|im_end|>'
# (248046). Left alone - a pad token distinct from EOS is strictly better here.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"  Tokenizer had no pad_token; set pad = eos ({tokenizer.eos_token!r})")
model.config.pad_token_id = tokenizer.pad_token_id
if getattr(model, "generation_config", None) is not None:
    model.generation_config.pad_token_id = tokenizer.pad_token_id

# ---- Attach the SFT adapter, trainable ----
print(f"Attaching SFT LoRA: {SFT_LORA_PATH}")
model = PeftModel.from_pretrained(model, str(SFT_LORA_PATH), is_trainable=True)

# ============ VERIFY THE INHERITED LoRA SCOPE ============
# DPO does NOT call get_peft_model - it continues the SFT adapter and inherits its module set
# exactly. If the SFT run leaked adapters into the vision tower or the MTP head, this run inherits
# the leak, spends hours on it, and produces an adapter vLLM refuses to load. This CANNOT be fixed
# here; it must be fixed at SFT. Catch it now, before the reference-logprob precompute.
from collections import Counter as _Counter


def _zone_of(param_name):
    """Return the non-language sub-model a parameter belongs to, or None."""
    n = param_name.replace("base_model.model.", "", 1)
    if n.startswith("mtp.") or ".mtp." in n:
        return "mtp head"
    if ".visual." in n or n.startswith("visual.") or "vision_tower" in n:
        return "vision tower"
    if "audio_tower" in n:
        return "audio tower"
    if "multi_modal_projector" in n or "embed_vision" in n or "embed_audio" in n:
        return "mm projector"
    return None


_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})
if not _adapted:
    raise RuntimeError(
        f"No LoRA modules found after attaching {SFT_LORA_PATH}. The adapter did not load."
    )

_stray = [n for n in _adapted if _zone_of(n)]
if _stray:
    raise RuntimeError(
        f"The SFT adapter has {len(_stray)} LoRA module(s) outside the language model "
        f"(e.g. {_stray[:3]}). The scope is INHERITED and cannot be fixed at DPO. Re-run the SFT "
        "notebook with finetune_vision_layers=False. See docs/multimodal_and_hybrid_base_models.md."
    )

_unfrozen, _zone_params = {}, _Counter()
for _n, _p in model.named_parameters():
    _z = _zone_of(_n)
    if _z is None:
        continue
    _zone_params[_z] += 1
    if _p.requires_grad:
        _unfrozen.setdefault(_z, []).append(_n)
if _unfrozen:
    raise RuntimeError(
        f"{sum(len(v) for v in _unfrozen.values())} tower/MTP parameter(s) are trainable "
        f"(e.g. {[n for v in _unfrozen.values() for n in v][:3]}). This is a text-only fine-tune."
    )

# The adapter must actually be TRAINABLE. PeftModel defaults to is_trainable=False, which loads
# the LoRA in inference mode - training would then run with zero trainable parameters and produce
# an adapter byte-identical to the SFT one, silently, after the full runtime.
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
if trainable == 0:
    raise RuntimeError(
        "The attached SFT adapter has zero trainable parameters. DPO would run for hours and "
        "change nothing. Check that PeftModel.from_pretrained was called with is_trainable=True."
    )

# Cross-check against what the SFT run recorded, so a mismatched adapter is caught here.
_sft_modules = _sft_sentinel.get("adapted_modules")
if _sft_modules is not None and _sft_modules != len(_adapted):
    raise RuntimeError(
        f"Adapter has {len(_adapted)} modules but the SFT sentinel recorded {_sft_modules}. "
        "The adapter directory does not match the run that wrote the sentinel."
    )

print("\nModel + SFT LoRA loaded")
print(f"  Base:             {BASE_LLM}")
print(f"  Adapter:          {SFT_LORA_PATH}")
print(f"  Adapted modules:  {len(_adapted)} (all inside the language model)")
for _z, _c in sorted(_zone_params.items()):
    print(f"  {_z:<16} {_c:>5} params  FROZEN")
print(f"  Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")
print(f"  Grad checkpoint:  True (standard recompute, no pinned-host offload)")
print(f"  Precision:        4-bit QLoRA")
print(f"  Max seq len:      {MAX_SEQ_LENGTH}")
print(f"  Tokenizer:        {type(tokenizer).__name__}"
      f"{' (unwrapped from ' + type(processor).__name__ + ')' if processor else ''}")
print(f"  GPU alloc:        {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading base: unsloth/Qwen3.5-4B
==((====))==  Unsloth 2026.8.22: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

  Gradient checkpointing: standard recompute (unsloth host-offload patch not active)
  (Extracted tokenizer from Processor - text-only DPO mode)
Attaching SFT LoRA: /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters

Model + SFT LoRA loaded
  Base:             unsloth/Qwen3.5-4B
  Adapter:          /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit/lora_adapters
  Adapted modules:  128 (all inside the language model)
  vision tower       297 params  FROZEN
  Trainable params: 42,467,328 / 2,646,192,640 (1.6048%)
  Grad checkpoint:  True (standard recompute, no pinned-host offload)
  Precision:        4-bit QLoRA
  Max seq len:      2048
  Tokenizer:        TokenizersBackend (unwrapped from Qwen3VLProcessor)
  GPU alloc:        3.5 GB


## 5. Validate & Format DPO Dataset (persistent cache)

Convert conversational `chosen` / `rejected` records into the `{prompt, chosen, rejected}` text
format `DPOTrainer` expects, and cache the result with a fingerprint so a kernel restart skips it.

Validation rejects a pair when: it is not a 3-message conversation, the prompt halves of `chosen`
and `rejected` differ, either final message is not an assistant turn, either response is empty, or
the two responses are identical. Those are all cases where the pair teaches nothing or teaches the
wrong thing.

Prompts render with `enable_thinking=False`, matching how the SFT data was built.

In [5]:
import hashlib, json, os, random, shutil
from collections import defaultdict
from pathlib import Path
from datasets import Dataset as HFDataset, load_from_disk

FORMATTED_CACHE_DIR = TRAIN_DIR / "formatted_dpo_dataset_cache"
FORMATTED_DATASET_DIR = FORMATTED_CACHE_DIR / "dataset"
FORMATTED_MANIFEST = FORMATTED_CACHE_DIR / "manifest.json"
FORMATTED_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _file_sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


# A content hash rather than an mtime: an edited-in-place dataset can never be silently reused.
format_config = {
    "cache_version": 1,
    "dpo_data_file": str(DPO_DATA_FILE),
    "dpo_data_sha256": _file_sha256(DPO_DATA_FILE),
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "dpo_max_pairs": DPO_MAX_PAIRS,
    "enable_thinking_training": False,
    "tokenizer_class": type(text_tokenizer).__name__,
    "tokenizer_name": BASE_LLM,
    "tokenizer_vocab_size": len(text_tokenizer),
}
format_fingerprint = hashlib.sha256(
    json.dumps(format_config, sort_keys=True).encode("utf-8")
).hexdigest()

_manifest = json.loads(FORMATTED_MANIFEST.read_text()) if FORMATTED_MANIFEST.exists() else None
dpo_dataset = None

if (not FORCE_REFORMAT and _manifest and _manifest.get("fingerprint") == format_fingerprint
        and FORMATTED_DATASET_DIR.exists()):
    dpo_dataset = load_from_disk(str(FORMATTED_DATASET_DIR))
    print(f"Reusing formatted DPO dataset cache: {FORMATTED_DATASET_DIR} ({len(dpo_dataset)} rows)")
elif _manifest:
    _diff = [k for k in format_config if _manifest.get("config", {}).get(k) != format_config[k]]
    print(f"Formatted-dataset cache is stale (changed: {_diff}) - rebuilding")

if dpo_dataset is None:
    errors = []
    formatted_pairs = []
    skipped_prompt_too_long = 0
    skipped_total_too_long = 0

    for index, pair in enumerate(raw_pairs):
        chosen_msgs = pair.get("chosen", [])
        rejected_msgs = pair.get("rejected", [])

        if len(chosen_msgs) < 3 or len(rejected_msgs) < 3:
            errors.append(f"pair {index}: too few messages "
                          f"(chosen={len(chosen_msgs)}, rejected={len(rejected_msgs)})")
            continue
        if chosen_msgs[:-1] != rejected_msgs[:-1]:
            errors.append(f"pair {index}: chosen/rejected prompt halves differ")
            continue
        if chosen_msgs[-1].get("role") != "assistant" or rejected_msgs[-1].get("role") != "assistant":
            errors.append(f"pair {index}: final messages must both be assistant turns")
            continue

        chosen_response = chosen_msgs[-1].get("content", "")
        rejected_response = rejected_msgs[-1].get("content", "")
        if not chosen_response.strip() or not rejected_response.strip():
            errors.append(f"pair {index}: empty chosen or rejected response")
            continue
        if chosen_response.strip() == rejected_response.strip():
            errors.append(f"pair {index}: chosen and rejected responses are identical")
            continue

        prompt_text = text_tokenizer.apply_chat_template(
            chosen_msgs[:-1],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,   # always False for TRAINING, matching the SFT notebook
        )

        p_len = len(text_tokenizer(prompt_text, add_special_tokens=False)["input_ids"])
        c_len = len(text_tokenizer(chosen_response, add_special_tokens=False)["input_ids"])
        r_len = len(text_tokenizer(rejected_response, add_special_tokens=False)["input_ids"])

        if p_len > MAX_PROMPT_LENGTH:
            skipped_prompt_too_long += 1
            continue
        if p_len + max(c_len, r_len) > MAX_SEQ_LENGTH:
            skipped_total_too_long += 1
            continue

        formatted_pairs.append({
            "prompt": prompt_text,
            "chosen": chosen_response,
            "rejected": rejected_response,
            "source": pair.get("source", "unknown"),
            "persona": pair.get("persona", "unknown"),
        })

    if errors:
        print(f"Validation errors: {len(errors)}")
        for error in errors[:10]:
            print(f"  {error}")
        if len(errors) > 10:
            print(f"  ... and {len(errors) - 10} more")
    else:
        print(f"All {len(raw_pairs):,} pairs passed structural validation")

    print(f"Length filter: dropped {skipped_prompt_too_long} for prompt > {MAX_PROMPT_LENGTH}, "
          f"{skipped_total_too_long} for total > {MAX_SEQ_LENGTH}")

    if not formatted_pairs:
        raise RuntimeError("No DPO pairs remain after validation/filtering.")

    # These limits were measured against this exact corpus and drop nothing. If a regenerated
    # dataset starts losing rows here, the run silently trains on a biased subset - the longest
    # (usually most detailed) rejected responses go first. Surface it rather than absorb it.
    _dropped = skipped_prompt_too_long + skipped_total_too_long
    _drop_rate = _dropped / len(raw_pairs)
    if _drop_rate > 0.02:
        raise RuntimeError(
            f"Length filter dropped {_dropped}/{len(raw_pairs)} pairs ({100 * _drop_rate:.1f}%), "
            f"above the 2% tolerance. These limits were set from a measurement where nothing was "
            f"dropped, so the dataset has changed. Re-measure and raise MAX_SEQ_LENGTH "
            f"({MAX_SEQ_LENGTH}) / MAX_PROMPT_LENGTH ({MAX_PROMPT_LENGTH}) in Cell 1."
        )
    _err_rate = len(errors) / len(raw_pairs)
    if _err_rate > 0.05:
        raise RuntimeError(
            f"{len(errors)}/{len(raw_pairs)} pairs ({100 * _err_rate:.1f}%) failed structural "
            "validation. Regenerate the DPO dataset before training."
        )

    # Cap, stratified proportionally by rejection source so the mix of failure modes is preserved.
    if DPO_MAX_PAIRS and len(formatted_pairs) > DPO_MAX_PAIRS:
        random.seed(3407)
        by_source = defaultdict(list)
        for pair in formatted_pairs:
            by_source[pair["source"]].append(pair)
        sampled = []
        total = len(formatted_pairs)
        for source, items in by_source.items():
            take = min(max(1, round(DPO_MAX_PAIRS * len(items) / total)), len(items))
            sampled.extend(random.sample(items, take))
        random.shuffle(sampled)
        formatted_pairs = sampled[:DPO_MAX_PAIRS]
        print(f"Capped to {len(formatted_pairs):,} pairs (stratified by source)")

    dpo_dataset = HFDataset.from_list(formatted_pairs).shuffle(seed=3407)

    if FORMATTED_DATASET_DIR.exists():
        shutil.rmtree(FORMATTED_DATASET_DIR)
    dpo_dataset.save_to_disk(str(FORMATTED_DATASET_DIR))
    _tmp_manifest = FORMATTED_MANIFEST.with_suffix(".json.tmp")
    _tmp_manifest.write_text(json.dumps({
        "status": "complete",
        "fingerprint": format_fingerprint,
        "config": format_config,
        "rows": len(dpo_dataset),
    }, indent=2))
    os.replace(_tmp_manifest, FORMATTED_MANIFEST)
    print(f"Saved formatted DPO dataset cache: {FORMATTED_DATASET_DIR}")

# ---- Report on whatever we ended up with, cached or fresh ----
_p_tok = [len(text_tokenizer(p, add_special_tokens=False)["input_ids"]) for p in dpo_dataset["prompt"]]
_c_tok = [len(text_tokenizer(c, add_special_tokens=False)["input_ids"]) for c in dpo_dataset["chosen"]]
_r_tok = [len(text_tokenizer(r, add_special_tokens=False)["input_ids"]) for r in dpo_dataset["rejected"]]

print(f"\nFormatted DPO dataset: {len(dpo_dataset):,} pairs")
print(f"  Columns: {dpo_dataset.column_names}")
print(f"  Source mix:  {dict(Counter(dpo_dataset['source']))}")
print(f"  Token lengths (max): prompt={max(_p_tok)}  chosen={max(_c_tok)}  rejected={max(_r_tok)}  "
      f"prompt+resp={max(p + max(c, r) for p, c, r in zip(_p_tok, _c_tok, _r_tok))}")
print(f"  Limits:              prompt<={MAX_PROMPT_LENGTH}  total<={MAX_SEQ_LENGTH}")

sample = dpo_dataset[0]
print("\nSample formatted pair:")
print(f"  PROMPT (tail):  ...{sample['prompt'][-220:]!r}")
print(f"  CHOSEN:         {sample['chosen'][:180]}...")
print(f"  REJECTED:       {sample['rejected'][:180]}...")

del raw_pairs

All 3,564 pairs passed structural validation
Length filter: dropped 0 for prompt > 1024, 0 for total > 2048


Saving the dataset (0/1 shards):   0%|          | 0/3564 [00:00<?, ? examples/s]

Saved formatted DPO dataset cache: /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/train/formatted_dpo_dataset_cache/dataset

Formatted DPO dataset: 3,564 pairs
  Columns: ['prompt', 'chosen', 'rejected', 'source', 'persona']
  Source mix:  {'shallow_platitude': 1200, 'voice_drift': 1164, 'scripture_fabrication': 1200}
  Token lengths (max): prompt=491  chosen=826  rejected=1048  prompt+resp=1506
  Limits:              prompt<=1024  total<=2048

Sample formatted pair:
  PROMPT (tail):  ..."e.<|im_end|>\n<|im_start|>user\nLooking back on the moment you confessed 'I do not know, my lord,' how did that honest admission shape your trust in God's unfolding plan?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
  CHOSEN:         I lifted my eyes and saw a golden lampstand fed by living trees, its seven flames trembling not with the wind but with the breath of God—and in that moment, my ignorance became an ...
  REJECTED:       Admitting one's lack of unders

## 6. DPO Trainer Setup

Two model-specific fixes are applied here, both verified against `trl 0.24.0` source in this
container rather than carried over from the 27B notebooks:

**The `model_type` swap is load-bearing.** `DPOTrainer.__init__` computes
`self.is_vision_model = model.config.model_type in MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES`
(dpo_trainer.py:359), then chooses `tokenize_row` vs `process_row` from it while preparing the
dataset — still inside `__init__` (line 667). `qwen3_5` **is** in that mapping; `qwen3_5_text` is
not. So the config is swapped to the nested text `model_type` for the duration of the constructor
and restored in a `finally`. Assigning `trainer.is_vision_model = False` afterwards, as the older
notebooks do, is too late to change how the dataset was tokenized.

**`precompute_ref_log_probs=False` is deliberate.** TRL's built-in precompute is one-shot and
in-memory — a crash partway through discards all of it. The next cell does the same work in
resumable shards on disk.

The cell finishes by asserting the dataset actually came out of the text path, so neither fix can
fail silently.

In [6]:
import math
from trl import DPOTrainer, DPOConfig

effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = math.ceil(len(dpo_dataset) / effective_batch)
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, int(max_steps * WARMUP_RATIO))

print("DPO Training Plan")
print(f"  DPO pairs:       {len(dpo_dataset):,}")
print(f"  Effective batch: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch}")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total steps:     {max_steps}")
print(f"  Warmup steps:    {warmup_steps}")
print(f"  DPO beta:        {DPO_BETA}  |  loss: {LOSS_TYPE}")

TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# Unsloth-patched PEFT models can lack the `warnings_issued` dict the HF Trainer expects.
if not hasattr(model, "warnings_issued"):
    object.__setattr__(model, "warnings_issued", {})
if hasattr(model, "base_model"):
    if not hasattr(model.base_model, "warnings_issued"):
        object.__setattr__(model.base_model, "warnings_issued", {})
    if hasattr(model.base_model, "model") and not hasattr(model.base_model.model, "warnings_issued"):
        model.base_model.model.warnings_issued = {}

# ---- Force the TEXT tokenization path for the duration of DPOTrainer.__init__ ----
# VERIFIED in trl 0.24.0 dpo_trainer.py:
#   line 359: self.is_vision_model = model.config.model_type in MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES
#   line 667: self.tokenize_row if not self.is_vision_model else self.process_row
# and in transformers: 'qwen3_5' IS in that mapping, 'qwen3_5_text' is NOT. process_row expects
# image inputs; our dataset is text-only. The swap must be in place while the constructor runs -
# setting trainer.is_vision_model afterwards does not re-tokenize anything.
_orig_model_type = getattr(model.config, "model_type", None)
_text_model_type = getattr(getattr(model.config, "text_config", None), "model_type", None)

if _text_model_type and _text_model_type != _orig_model_type:
    model.config.model_type = _text_model_type
    print(f"\n  model.config.model_type: {_orig_model_type!r} -> {_text_model_type!r} "
          f"(restored after __init__)")
else:
    print(f"\n  model.config.model_type ({_orig_model_type!r}) left unchanged")
print(f"  processing_class: {type(text_tokenizer).__name__}")

try:
    trainer = DPOTrainer(
        model=model,
        # None: with a PEFT model TRL uses the base with adapters disabled as the implicit
        # reference, so no second copy of the weights is allocated.
        ref_model=None,
        processing_class=text_tokenizer,
        train_dataset=dpo_dataset,
        args=DPOConfig(
            beta=DPO_BETA,
            loss_type=LOSS_TYPE,
            max_length=MAX_SEQ_LENGTH,
            max_prompt_length=MAX_PROMPT_LENGTH,
            # False on purpose - the persistent shard cache cell below does this resumably.
            precompute_ref_log_probs=False,
            precompute_ref_batch_size=REF_LOGPROBS_BATCH_SIZE,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_steps=warmup_steps,
            max_steps=max_steps,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            optim="adamw_8bit",
            weight_decay=0.0,
            seed=3407,
            gradient_checkpointing=True,
            # Pinned host memory is page-locked and unreclaimable, and on GB10 it comes out of the
            # same pool the model trains in.
            dataloader_pin_memory=False,
            output_dir=str(TRAIN_DIR),
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            logging_steps=5,
            report_to="none",
            dataset_num_proc=1,
        ),
    )
finally:
    # Restore unconditionally, including on failure - a half-swapped config would make every
    # later cell behave differently for reasons nothing records.
    if _orig_model_type is not None:
        model.config.model_type = _orig_model_type

# ---- Assert the text path actually ran ----
# If process_row had been used, the prepared dataset would carry vision columns and would be
# missing the text ones. Checking the artifact beats trusting the mechanism.
_cols = set(trainer.train_dataset.column_names)
_required = {"prompt_input_ids", "chosen_input_ids", "rejected_input_ids"}
_vision_leak = {"pixel_values", "image_sizes", "pixel_attention_mask"} & _cols
if not _required.issubset(_cols):
    raise RuntimeError(
        f"Prepared dataset is missing {sorted(_required - _cols)}. DPOTrainer did not use the "
        f"text tokenization path. Columns: {sorted(_cols)}"
    )
if _vision_leak:
    raise RuntimeError(
        f"Prepared dataset carries vision columns {sorted(_vision_leak)} - process_row() ran "
        "instead of tokenize_row(). The model_type swap did not take effect."
    )

print("\nDPO Trainer configured")
print(f"  Prepared columns:         {sorted(_cols)}")
print(f"  Text path verified:       tokenize_row (no vision columns)")
print(f"  model.config.model_type:  {model.config.model_type!r} (restored)")
print(f"  precompute_ref_log_probs: False (persistent shard cache handles this)")
print(f"  gradient_checkpointing:   True")
print(f"  dataloader_pin_memory:    False")
print(f"  Checkpoints:              every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Precision:                {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

DPO Training Plan
  DPO pairs:       3,564
  Effective batch: 1 x 8 = 8
  Steps per epoch: 446
  Total steps:     446
  Warmup steps:    44
  DPO beta:        0.1  |  loss: sigmoid

  model.config.model_type: 'qwen3_5' -> 'qwen3_5_text' (restored after __init__)
  processing_class: TokenizersBackend


Extracting prompt in train dataset:   0%|          | 0/3564 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/3564 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3564 [00:00<?, ? examples/s]


DPO Trainer configured
  Prepared columns:         ['chosen_input_ids', 'persona', 'prompt', 'prompt_input_ids', 'rejected_input_ids', 'source']
  Text path verified:       tokenize_row (no vision columns)
  model.config.model_type:  'qwen3_5' (restored)
  precompute_ref_log_probs: False (persistent shard cache handles this)
  gradient_checkpointing:   True
  dataloader_pin_memory:    False
  Checkpoints:              every 50 steps, keep 3
  Precision:                bf16


## 7. Persistent Reference Logprob Cache

Precompute the frozen-reference log probabilities in **resumable shards on disk** before training.

**What the reference is here:** `ref_model=None` on a PEFT model, so TRL's `null_ref_context()` disables the adapter and runs the reference forward on the **bare 4-bit base** (dpo_trainer.py:915-924). No second copy of the weights is allocated, and the SFT adapter does not affect these values — they are a function of the base weights and the tokenized data. The cache is keyed on `base_llm` and lives under this model's own `train/` directory, so it can never be picked up by a run against a different base.

TRL's built-in precompute is one-shot and in-memory: a crash partway through throws all of it away.
This writes each completed shard atomically with a config fingerprint, so a failed run resumes from
the first missing shard instead of from zero. Shards are validated on load — a truncated or stale
shard is recomputed rather than silently trusted.

Verified on `trl 0.24.0`: `DataCollatorForPreference` uses `ref_chosen_logps` / `ref_rejected_logps`
when they are present in a row (dpo_trainer.py:159-179), and `_set_signature_columns_if_needed`
lists both so `_remove_unused_columns` does not prune them (line 807-814). Adding these columns is
therefore enough to skip the reference forward pass during training.

In [7]:
import hashlib, json, os, time
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_from_disk

REF_LOGPROBS_CACHE_DIR = TRAIN_DIR / "ref_logprobs_cache"
REF_LOGPROBS_SHARD_DIR = REF_LOGPROBS_CACHE_DIR / "shards"
REF_LOGPROBS_DATASET_DIR = REF_LOGPROBS_CACHE_DIR / "dataset"
REF_LOGPROBS_MANIFEST = REF_LOGPROBS_CACHE_DIR / "manifest.json"

REF_LOGPROBS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REF_LOGPROBS_SHARD_DIR.mkdir(parents=True, exist_ok=True)

# What these logprobs actually are, verified in trl 0.24.0 rather than assumed:
# with ref_model=None on a PEFT model, compute_ref_log_probs() enters null_ref_context()
# (dpo_trainer.py:915-924), which calls disable_adapter() and runs concatenated_forward on
# self.model with the LoRA switched OFF. So the reference is the BARE 4-bit base model - the SFT
# adapter contributes nothing to these values.
#
# They therefore depend on: the base weights + its quantization, the tokenized data, and the
# tokenization limits. `sft_lora_path` is keyed in as well, which is stricter than correctness
# requires: these numbers would in fact be valid across two different SFT adapters on the same
# base. That over-keying is deliberate - re-running SFT to the same path is the normal way this
# tree changes, and silently reusing a cache across a changed adapter directory is a worse failure
# than recomputing ~50 shards. It is NOT what makes the cache safe across base models; `base_llm`
# is, and TRAIN_DIR is per-MODEL_NAME_BASE on top of that.
_ref_cache_config = {
    "cache_version": 1,
    "base_llm": BASE_LLM,
    "sft_lora_path": str(SFT_LORA_PATH),
    "dpo_data_file": str(DPO_DATA_FILE),
    "formatted_fingerprint": format_fingerprint,
    "dataset_len": len(trainer.train_dataset),
    "dataset_columns": sorted(c for c in trainer.train_dataset.column_names if not c.startswith("ref_")),
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "tokenizer_class": type(text_tokenizer).__name__,
    "tokenizer_vocab_size": len(text_tokenizer),
    "shard_size": REF_LOGPROBS_SHARD_SIZE,
    "batch_size": REF_LOGPROBS_BATCH_SIZE,
}
_ref_cache_fingerprint = hashlib.sha256(
    json.dumps(_ref_cache_config, sort_keys=True).encode("utf-8")
).hexdigest()


def _read_ref_manifest():
    if not REF_LOGPROBS_MANIFEST.exists():
        return None
    try:
        return json.loads(REF_LOGPROBS_MANIFEST.read_text())
    except (OSError, json.JSONDecodeError):
        return None


def _load_valid_ref_shard(shard_path, shard_idx, start, end):
    """Load a shard, raising if it is unreadable, stale, or the wrong shape."""
    try:
        payload = torch.load(shard_path, map_location="cpu")
    except (EOFError, OSError, RuntimeError, ValueError) as exc:
        raise RuntimeError(f"unreadable shard: {exc}") from exc
    if not isinstance(payload, dict):
        raise RuntimeError("shard payload is not a dict")
    if payload.get("fingerprint") != _ref_cache_fingerprint:
        raise RuntimeError("stale fingerprint")
    if payload.get("shard_idx") != shard_idx:
        raise RuntimeError("shard index mismatch")
    if payload.get("start") != start or payload.get("end") != end:
        raise RuntimeError("shard range mismatch")
    expected_rows = end - start
    for column in ("ref_chosen_logps", "ref_rejected_logps"):
        values = payload.get(column)
        if not isinstance(values, torch.Tensor) or values.numel() != expected_rows:
            raise RuntimeError(f"invalid {column} ({expected_rows} rows expected)")
        if not torch.isfinite(values).all():
            raise RuntimeError(f"non-finite values in {column}")
    return payload


def _write_ref_manifest(status, completed_shards):
    manifest = {
        "status": status,
        "fingerprint": _ref_cache_fingerprint,
        "config": _ref_cache_config,
        "completed_shards": completed_shards,
        "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    tmp_path = REF_LOGPROBS_MANIFEST.with_suffix(".json.tmp")
    tmp_path.write_text(json.dumps(manifest, indent=2))
    os.replace(tmp_path, REF_LOGPROBS_MANIFEST)


_required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}
_manifest = _read_ref_manifest()
_cache_matches = _manifest is not None and _manifest.get("fingerprint") == _ref_cache_fingerprint

if _cache_matches and REF_LOGPROBS_DATASET_DIR.exists():
    cached_dataset = load_from_disk(str(REF_LOGPROBS_DATASET_DIR))
    if (len(cached_dataset) == len(trainer.train_dataset)
            and _required_ref_cols.issubset(cached_dataset.column_names)):
        trainer.train_dataset = cached_dataset
        trainer._precomputed_train_ref_log_probs = True
        print(f"Loaded persistent ref logprob cache: {REF_LOGPROBS_DATASET_DIR}")
    else:
        print("Ignoring stale ref logprob dataset cache: length or columns do not match")
        _cache_matches = False

if not _required_ref_cols.issubset(trainer.train_dataset.column_names):
    if not _cache_matches:
        # Fingerprint changed - every existing shard describes a different computation.
        _stale = list(REF_LOGPROBS_SHARD_DIR.glob("shard-*.pt"))
        if _stale:
            print(f"Config fingerprint changed - discarding {len(_stale)} stale shard(s)")
        for stale_shard in _stale:
            stale_shard.unlink()
        _write_ref_manifest("in_progress", [])

    num_rows = len(trainer.train_dataset)
    shard_ranges = [
        (start, min(start + REF_LOGPROBS_SHARD_SIZE, num_rows))
        for start in range(0, num_rows, REF_LOGPROBS_SHARD_SIZE)
    ]

    print("Persistent reference logprob cache")
    print(f"  Rows:       {num_rows:,}")
    print(f"  Shards:     {len(shard_ranges)} of up to {REF_LOGPROBS_SHARD_SIZE} rows")
    print(f"  Cache dir:  {REF_LOGPROBS_CACHE_DIR}")

    completed = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if shard_path.exists():
            try:
                _load_valid_ref_shard(shard_path, shard_idx, start, end)
            except RuntimeError as exc:
                print(f"  Recomputing shard {shard_idx} ({exc})")
                shard_path.unlink()
            else:
                completed.append(shard_idx)
                continue

        shard_dataset = trainer.train_dataset.select(range(start, end))
        shard_loader = DataLoader(
            shard_dataset,
            batch_size=REF_LOGPROBS_BATCH_SIZE,
            collate_fn=trainer.data_collator,
            num_workers=0,
            pin_memory=False,
            shuffle=False,
        )
        shard_loader = trainer.accelerator.prepare(shard_loader)

        ref_chosen_logps, ref_rejected_logps = [], []
        for padded_batch in tqdm(shard_loader,
                                 desc=f"Ref logprobs shard {shard_idx + 1}/{len(shard_ranges)}",
                                 leave=False):
            ref_chosen_logp, ref_rejected_logp = trainer.compute_ref_log_probs(padded_batch)
            ref_chosen_logp, ref_rejected_logp = trainer.accelerator.gather_for_metrics(
                (ref_chosen_logp, ref_rejected_logp)
            )
            ref_chosen_logps.append(ref_chosen_logp.float().cpu())
            ref_rejected_logps.append(ref_rejected_logp.float().cpu())

        payload = {
            "fingerprint": _ref_cache_fingerprint,
            "shard_idx": shard_idx,
            "start": start,
            "end": end,
            "ref_chosen_logps": torch.cat(ref_chosen_logps),
            "ref_rejected_logps": torch.cat(ref_rejected_logps),
        }
        # Write to a temp path then rename: a crash mid-write leaves the old shard or no shard,
        # never a half-written one that would be loaded as valid.
        tmp_shard_path = shard_path.with_suffix(".pt.tmp")
        torch.save(payload, tmp_shard_path)
        os.replace(tmp_shard_path, shard_path)
        completed.append(shard_idx)
        _write_ref_manifest("in_progress", completed)

        # Flush per shard, not per batch: relief from fragmentation without paying the
        # allocator re-acquisition cost on every step.
        gc_collected = None
        import gc as _gc
        gc_collected = _gc.collect()
        torch.cuda.empty_cache()
        trainer.accelerator.free_memory()

    all_chosen, all_rejected = [], []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if not shard_path.exists():
            raise RuntimeError(f"Missing ref logprob shard: {shard_path}")
        payload = _load_valid_ref_shard(shard_path, shard_idx, start, end)
        all_chosen.append(payload["ref_chosen_logps"])
        all_rejected.append(payload["ref_rejected_logps"])

    ref_chosen_values = torch.cat(all_chosen).numpy()
    ref_rejected_values = torch.cat(all_rejected).numpy()
    if len(ref_chosen_values) != num_rows or len(ref_rejected_values) != num_rows:
        raise RuntimeError(
            f"Ref logprob cache has {len(ref_chosen_values)} rows but the dataset has {num_rows}."
        )

    train_dataset_with_ref = trainer.train_dataset
    for ref_col in ("ref_chosen_logps", "ref_rejected_logps"):
        if ref_col in train_dataset_with_ref.column_names:
            train_dataset_with_ref = train_dataset_with_ref.remove_columns(ref_col)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_chosen_logps", ref_chosen_values)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_rejected_logps", ref_rejected_values)

    if REF_LOGPROBS_DATASET_DIR.exists():
        import shutil
        shutil.rmtree(REF_LOGPROBS_DATASET_DIR)
    train_dataset_with_ref.save_to_disk(str(REF_LOGPROBS_DATASET_DIR))
    trainer.train_dataset = train_dataset_with_ref
    trainer._precomputed_train_ref_log_probs = True
    _write_ref_manifest("complete", list(range(len(shard_ranges))))
    print(f"Saved persistent ref logprob dataset cache: {REF_LOGPROBS_DATASET_DIR}")

# Sanity-check the cached values before committing hours to training on them. Reference logprobs
# are sums of log probabilities over response tokens: always negative, never zero, never NaN.
import numpy as _np
_rc = _np.asarray(trainer.train_dataset["ref_chosen_logps"], dtype=float)
_rr = _np.asarray(trainer.train_dataset["ref_rejected_logps"], dtype=float)
if not (_np.isfinite(_rc).all() and _np.isfinite(_rr).all()):
    raise RuntimeError("Reference logprob cache contains non-finite values.")
if (_rc >= 0).any() or (_rr >= 0).any():
    raise RuntimeError("Reference logprob cache contains non-negative logprobs - these are wrong.")

print(f"\nRef logprob columns ready: {sorted(_required_ref_cols)}")
print(f"  chosen   mean={_rc.mean():9.2f}  min={_rc.min():9.2f}  max={_rc.max():9.2f}")
print(f"  rejected mean={_rr.mean():9.2f}  min={_rr.min():9.2f}  max={_rr.max():9.2f}")

Persistent reference logprob cache
  Rows:       3,564
  Shards:     56 of up to 64 rows
  Cache dir:  /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/train/ref_logprobs_cache


Ref logprobs shard 1/56:   0%|          | 0/64 [00:00<?, ?it/s]

Ref logprobs shard 2/56:   0%|          | 0/64 [00:00<?, ?it/s]

Ref logprobs shard 3/56:   0%|          | 0/64 [00:00<?, ?it/s]

Ref logprobs shard 4/56:   0%|          | 0/64 [00:00<?, ?it/s]

...

Ref logprobs shard 54/56:   0%|          | 0/64 [00:00<?, ?it/s]

Ref logprobs shard 55/56:   0%|          | 0/64 [00:00<?, ?it/s]

Ref logprobs shard 56/56:   0%|          | 0/44 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/3564 [00:00<?, ? examples/s]

Saved persistent ref logprob dataset cache: /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/train/ref_logprobs_cache/dataset

Ref logprob columns ready: ['ref_chosen_logps', 'ref_rejected_logps']
  chosen   mean=  -914.22  min= -2352.00  max=   -77.50
  rejected mean=  -542.85  min= -1672.00  max=   -32.75


## 8. Train

Auto-resumes from the newest `checkpoint-*` in the train directory, and refuses to start at all if
the reference-logprob columns are missing.

**What to watch:** loss starts near `0.693` (`-log 0.5`, the value at zero preference margin) and
should fall toward roughly `0.40–0.55`. `rewards/margins` should widen as chosen pulls ahead of
rejected. A loss stuck flat at 0.693 means the policy is not separating the pairs.

In [8]:
import gc, os
from transformers import TrainerCallback
from transformers.trainer_utils import get_last_checkpoint


class PeriodicMemoryCleanup(TrainerCallback):
    """Collect dead objects and return cached CUDA blocks to the allocator.

    Runs every `every` optimizer steps, on step end only. Flushing on every step - or on both
    step begin and step end - forces the allocator to re-acquire blocks and measurably slows
    training; the allocator only needs periodic relief from fragmentation.
    """

    def __init__(self, every=50):
        self.every = max(1, int(every))

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0:
            gc.collect()
            torch.cuda.empty_cache()
        return control


trainer.add_callback(PeriodicMemoryCleanup(CLEANUP_STEPS))

required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}
if not required_ref_cols.issubset(trainer.train_dataset.column_names):
    raise RuntimeError("Run the persistent reference-logprob cache cell (Cell 7) before training.")

# Guard against the silent no-op: a non-trainable adapter would run the full schedule and save an
# adapter byte-identical to the SFT one.
_trainable_now = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
if _trainable_now == 0:
    raise RuntimeError("Model has zero trainable parameters. DPO would change nothing.")

print("DPO training starting")
print(f"  Trainable params:  {_trainable_now:,}")
print(f"  Memory cleanup:    every {CLEANUP_STEPS} steps")
print(f"  Expect loss to fall from ~0.693 toward 0.40-0.55, and rewards/margins to widen.\n")

_ckpt_dir = trainer.args.output_dir
last_ckpt = get_last_checkpoint(_ckpt_dir) if os.path.isdir(_ckpt_dir) else None
if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    result = trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("No checkpoint found - starting from scratch.")
    result = trainer.train()

print("\nDPO training complete")
print(f"  Final loss:  {result.training_loss:.4f}")
print(f"  Total steps: {result.global_step}")
if result.training_loss > 0.68:
    print("  NOTE: loss barely moved from 0.693. The policy is not separating chosen from "
          "rejected - check the reward margins in the log history before shipping this adapter.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


DPO training starting
  Trainable params:  42,467,328
  Memory cleanup:    every 50 steps
  Expect loss to fall from ~0.693 toward 0.40-0.55, and rewards/margins to widen.

No checkpoint found - starting from scratch.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,564 | Num Epochs = 1 | Total steps = 446
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 42,467,328 of 4,581,732,864 (0.93% trained)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,564 | Num Epochs = 1 | Total steps = 446
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 42,467,328 of 4,581,732,864 (0.93% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.000000,25.477734,-4.177102,1.000000,29.654837,-614.022644,-607.296021,-1.540548,-1.002388
10,0.000000,25.229053,-3.908886,1.000000,29.137939,-665.959473,-557.263855,-1.488235,-1.008120
15,0.000564,25.185556,-3.701580,1.000000,28.887135,-593.031921,-636.503296,-1.574154,-1.168366
20,0.000000,26.745438,-4.351054,1.000000,31.096491,-615.970581,-590.591797,-1.563432,-1.110033
25,0.000000,27.122272,-4.013800,1.000000,31.136072,-631.577271,-599.825500,-1.543250,-1.061887
30,0.000000,25.199097,-4.026361,1.000000,29.225458,-655.808960,-549.088562,-1.504511,-0.994909
35,0.000000,24.575022,-4.197089,1.000000,28.772114,-640.899780,-665.508423,-1.441291,-1.043178
40,0.000001,25.034405,-3.875541,1.000000,28.909946,-625.255981,-643.255432,-1.495286,-1.059686
45,0.000001,26.581116,-4.651165,1.000000,31.232279,-634.288879,-646.849182,-1.550683,-1.036336
50,0.000001,25.525692,-2.379459,1.000000,27.905151,-615.493042,-479.319641,-1.498588,-1.045274


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/train/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/train/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/train/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/train/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/train/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/train/checkpoint-150/tokenizer_config.


DPO training complete
  Final loss:  0.0000
  Total steps: 446


## 9. Save DPO LoRA Adapter

Saves the combined SFT+DPO LoRA (still a single adapter against the base), the Processor files, the
persona system prompts, and a `complete.json` sentinel. Load this directly on top of
`unsloth/Qwen3.5-4B` or any compatible quantization of the same base.

In [9]:
import json
from datetime import datetime, timezone

LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving DPO LoRA adapter to {LORA_OUTPUT_DIR}...")
model.save_pretrained(str(LORA_OUTPUT_DIR))
# Save the Processor when one was unwrapped - save_pretrained on a Processor writes the tokenizer
# files too, so this is a superset of tokenizer.save_pretrained().
(processor or tokenizer).save_pretrained(str(LORA_OUTPUT_DIR))

prompts_path = LORA_OUTPUT_DIR / "persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

# Machine-readable completion sentinel (docs/README.md: "Final model-producing notebooks must
# write a machine-readable completion sentinel").
sentinel = {
    "status": "complete",
    "stage": "dpo",
    "model_name": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "sft_model_name": SFT_MODEL_NAME_BASE,
    "sft_adapter": str(SFT_LORA_PATH),
    "sft_global_step": _sft_sentinel.get("global_step"),
    "input_data_file": str(DPO_DATA_FILE),
    "output_dir": str(TRAIN_DIR),
    "lora_output_dir": str(LORA_OUTPUT_DIR),
    "last_checkpoint": str(last_ckpt) if last_ckpt else None,
    "global_step": int(result.global_step),
    "max_steps": int(max_steps),
    "final_loss": float(result.training_loss),
    "dpo_pairs": int(len(dpo_dataset)),
    "dpo_max_pairs": DPO_MAX_PAIRS,
    "beta": DPO_BETA,
    "loss_type": LOSS_TYPE,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "adapted_modules": len(_adapted),
    "formatted_cache_fingerprint": format_fingerprint,
    "ref_logprob_cache_fingerprint": _ref_cache_fingerprint,
    "personas": sorted(persona_system_prompts),
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
with open(LORA_OUTPUT_DIR / "complete.json", "w") as f:
    json.dump(sentinel, f, indent=2)

print(f"\nDPO LoRA adapter saved")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  Sentinel:       {LORA_OUTPUT_DIR / 'complete.json'} "
      f"(step {result.global_step}, loss {result.training_loss:.4f})")
print(f"  System prompts: {prompts_path} ({len(persona_system_prompts)} personas)")

total_size = 0
print(f"\nAdapter contents:")
for path in sorted(LORA_OUTPUT_DIR.iterdir()):
    size = path.stat().st_size
    total_size += size
    print(f"  {path.name:40s} {size / 1024 / 1024:8.1f} MB")
print(f"  {'TOTAL':40s} {total_size / 1024 / 1024:8.1f} MB")

print("\nAudit the adapter scope before shipping:")
print("  docker exec unsloth-notebook python /workspace/training/docs/audit_adapters.py")

Saving DPO LoRA adapter to /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/lora_adapters...


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/lora_adapters/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/lora_adapters/tokenizer_config.json.



DPO LoRA adapter saved
  Adapters:       /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/lora_adapters
  Sentinel:       /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/lora_adapters/complete.json (step 446, loss 0.0000)
  System prompts: /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/lora_adapters/persona_system_prompts.json (26 personas)

Adapter contents:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   162.0 MB
  chat_template.jinja                           0.0 MB
  complete.json                                 0.0 MB
  persona_system_prompts.json                   0.0 MB
  processor_config.json                         0.0 MB
  tokenizer.json                               19.1 MB
  tokenizer_config.json                         0.0 MB
  TOTAL                                       181.2 MB

Audit the a

## 10. Quick Evaluation

Smoke test a few personas with their own system prompts. The DPO model should keep the persona voice
while avoiding the generic, fabricated, and out-of-voice answers the rejected samples represent.

Uses `ENABLE_THINKING` from Cell 1, so the rendered prompt matches what training saw.

In [10]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

test_personas = list(persona_system_prompts.keys())[:4]
print(f"Quick evaluation across {len(test_personas)} personas "
      f"(thinking={'ON' if ENABLE_THINKING else 'OFF'})\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )
    # Passed by keyword: required if this ever runs against a Processor, whose first positional
    # parameter is `images`, not `text`.
    inputs = tokenizer(text=text, return_tensors="pt").to(model.device)

    print("=" * 70)
    print(f"PERSONA: {persona_key.upper()}")
    print(f"Q: {TEST_PROMPT}")
    print("A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024 if ENABLE_THINKING else 384,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        top_k=GEN_TOP_K,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print("\n")

del inputs, outputs

Quick evaluation across 4 personas (thinking=OFF)

PERSONA: SOLOMON
Q: I am struggling with forgiveness. What does Scripture teach about forgiving others?
A: The heart that refuses to let go becomes a prison for both the jailer and the captive.  

My son, when you stand before the Lord with a burdened soul, know this: He who forgives others is not made weak, but made strong. For in the day of judgment, He will not count your failures—only your faithfulness in bearing another’s.  

I have seen the wise man gather knowledge like a man gathers pearls; so too does he who learns to forgive gather grace like rain upon dry ground. But he who grudges his brother becomes a well without water—empty, cracked, yielding nothing but dust.  

Forgiveness is not forgetting. It is not forgetting the wound, but refusing to dwell in it. It is to lift the other’s face even when your own are stained. The Lord says: *I will not deal with you according to your sins, nor return you to your own way.* But only 

## 11. Verify Adapter Reload (Cold, from Disk)

Reload the saved DPO adapter cold to confirm it is self-contained and portable, and audit the saved
`adapter_model.safetensors` keys directly. The tensor keys are ground truth — a notebook drifts from
what produced its output, the artifact does not.

In [11]:
import gc, json, struct
from collections import Counter
from pathlib import Path

del model, tokenizer, trainer, dpo_dataset
gc.collect()
torch.cuda.empty_cache()

print(f"Reloading adapter from: {LORA_OUTPUT_DIR}")
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

if hasattr(tokenizer2, "tokenizer"):
    tokenizer2 = tokenizer2.tokenizer

with open(LORA_OUTPUT_DIR / "persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)
with open(LORA_OUTPUT_DIR / "complete.json") as f:
    reloaded_sentinel = json.load(f)

# ---- Audit the SAVED artifact, not this notebook's variables ----
_sf = LORA_OUTPUT_DIR / "adapter_model.safetensors"
with open(_sf, "rb") as f:
    _hdr = json.loads(f.read(struct.unpack("<Q", f.read(8))[0]))
_fam = Counter()
for k in _hdr:
    if k == "__metadata__":
        continue
    n = k.replace("base_model.model.", "")
    _fam["VISION" if ("vision_tower" in n or ".visual." in n)
         else "AUDIO" if "audio_tower" in n
         else "MTP" if (n.startswith("mtp.") or ".mtp." in n)
         else "language"] += 1
print(f"\nSaved adapter tensor families: {dict(_fam)}")
if set(_fam) != {"language"}:
    raise RuntimeError(
        f"Saved DPO adapter contains non-language tensors: {dict(_fam)}. "
        "vLLM will refuse this adapter. Do not ship it."
    )

test_key = list(reloaded_prompts.keys())[0]
messages = [
    {"role": "system", "content": reloaded_prompts[test_key]},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)
inputs = tokenizer2(text=text, return_tensors="pt").to(model2.device)
outputs = model2.generate(
    **inputs,
    max_new_tokens=1024 if ENABLE_THINKING else 384,
    temperature=GEN_TEMPERATURE,
    top_p=GEN_TOP_P,
    top_k=GEN_TOP_K,
    do_sample=True,
)
response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (persona: {test_key}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\nAdapter loads cleanly from disk and carries only language-model tensors.")
print(f"  Sentinel: SFT step {reloaded_sentinel['sft_global_step']} -> "
      f"DPO step {reloaded_sentinel['global_step']}, "
      f"loss {reloaded_sentinel['final_loss']:.4f}, "
      f"{reloaded_sentinel['dpo_pairs']} pairs")
print("\nReady for deployment via vLLM.")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

Reloading adapter from: /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/lora_adapters
==((====))==  Unsloth 2026.8.22: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]


Saved adapter tensor families: {'language': 256}

ADAPTER RELOAD TEST (persona: solomon):
  Q: I am struggling with forgiveness. What does Scripture teach about forgiving others?
  A: A garden may be overgrown with thorns, yet still bloom with fragrance — so it is with the heart that bears grudges: the more it clings to bitterness, the less it breathes the scent of mercy.

My son, to forgive is not to forget, but to release — as the river gives way to the sea without counting the stones it has carried. The Lord has taught me: when you lift your hand in anger, it is a sign that you are still bound to your enemy’s sin. But when you lay down that hand, you become a living sacrif

Adapter loads cleanly from disk and carries only language-model tensors.
  Sentinel: SFT step 544 -> DPO step 446, loss 0.0000, 3564 pairs

Ready for deployment via vLLM.


## 12. Export Merged Model to GGUF — optional

Merge the SFT+DPO LoRA into the base and export to GGUF for llama.cpp / Ollama / LM Studio / iOS
GGUF runners. At 4B this is genuinely comfortable on a phone.

**Not needed for the vLLM path** — the LoRA adapter saved in Cell 9 is the shipping artifact.

> **This path does not work on `Qwen3_5ForConditionalGeneration`.** The SFT run failed here with
> `Unsloth: Failed to convert model to GGUF` out of `unsloth_convert_hf_to_gguf.py` (its patched
> converter reports no supported TEXT/VISION architecture). **Use Section 13 instead** — it is the
> route that actually produced the shipped SFT GGUFs, and it also emits a standalone LoRA GGUF.

**Memory:** reloads the base at `load_in_4bit=False`, i.e. ~8.7 GB of bf16 weights, then merges.

In [12]:
# Compatibility shim for PEFT + GPTQModel during GGUF export.
# PEFT imports GPTQModel's older AWQ class name even when this adapter is not AWQ.
try:
    import gptqmodel.nn_modules.qlinear.gemm_awq as _gemm_awq
    if not hasattr(_gemm_awq, "AwqGEMMQuantLinear") and hasattr(_gemm_awq, "AwqGEMMLinear"):
        _gemm_awq.AwqGEMMQuantLinear = _gemm_awq.AwqGEMMLinear
        print("Patched GPTQModel AWQ class alias for PEFT compatibility.")
except Exception as exc:
    print(f"GPTQModel AWQ compatibility shim skipped: {exc}")

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


fatal: detected dubious ownership in repository at '/workspace/training/biblical'
To add an exception for this directory, call:

	git config --global --add safe.directory /workspace/training/biblical


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0
Transformers : 5.5.0
Torch        : 2.10.0a0+b558c986e8.nv25.11
Triton       : 3.4.0+gitc5d671f9


Patched GPTQModel AWQ class alias for PEFT compatibility.


In [13]:
# Reload the adapter fresh so we export from a clean state
import gc, torch
from pathlib import Path
from unsloth import FastLanguageModel

# Drop anything still holding weights from earlier cells. Each name is deleted independently so
# one already-absent name does not skip the rest.
for _name in ("model", "tokenizer", "trainer", "model2", "tokenizer2", "dpo_dataset"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

GGUF_OUTPUT_DIR = OUTPUT_BASE_DIR / "gguf"
GGUF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
)

# ---- Force the chat template's end-of-turn token as the GGUF EOS ----
# REQUIRED for this checkpoint, not defensive boilerplate. Verified in its config.json and
# tokenizer_config.json:
#     config.text_config.eos_token_id = 248044  ->  <|endoftext|>
#     tokenizer eos_token / template close       ->  <|im_end|> (248046)
# llama.cpp's converter reads eos_token_id from the NESTED text_config on
# Conditional-Generation models, so without this it bakes in <|endoftext|>, the model never stops
# on <|im_end|>, and it runs on into the next turn header. iOS GGUF runners then strip the leading
# <|im_start|> and render bare role text plus a hallucinated follow-up question.
_render = tokenizer_gguf.apply_chat_template(
    [{"role": "user", "content": "x"}, {"role": "assistant", "content": "y"}],
    tokenize=False,
)
_eot_candidates = ["<|im_end|>", "<end_of_turn>", "<|eot_id|>"]
eot_token = next((c for c in _eot_candidates if c in _render), None)
if eot_token is None:
    raise RuntimeError(f"Could not detect end-of-turn marker in chat template. Rendered: {_render!r}")

_tok = getattr(tokenizer_gguf, "tokenizer", tokenizer_gguf)
eot_id = _tok.convert_tokens_to_ids(eot_token)
if eot_id is None or eot_id == _tok.unk_token_id:
    raise RuntimeError(f"{eot_token!r} not in tokenizer vocab (got id={eot_id})")

_tok.eos_token = eot_token
model_gguf.config.eos_token_id = eot_id
# The one that actually matters: the nested text_config the converter reads from.
if getattr(model_gguf.config, "text_config", None) is not None:
    model_gguf.config.text_config.eos_token_id = eot_id
# generation_config.eos_token_id may be a list; collapse to the single chat-template EOS.
if getattr(model_gguf, "generation_config", None) is not None:
    model_gguf.generation_config.eos_token_id = eot_id

print(f"Forced GGUF EOS to {eot_token!r} (id={eot_id}); text_config default was 248044 (<|endoftext|>)")

# Pass ALL quant methods in one call so the LoRA->FP16 merge happens ONCE and llama.cpp quantizes
# from that single merged file. A loop would re-merge and re-write the FP16 GGUF per quant level.
QUANT_METHODS = ["q4_k_m", "q5_k_m", "q8_0"]

print(f"Exporting GGUF (single merge -> {len(QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    str(GGUF_OUTPUT_DIR),
    tokenizer_gguf,
    quantization_method=QUANT_METHODS,
)

print(f"\nGGUF export complete: {GGUF_OUTPUT_DIR}")
for f in sorted(GGUF_OUTPUT_DIR.glob("*.gguf")):
    print(f"  {f.name:60s} {f.stat().st_size / 1024 / 1024:>8.1f} MB")

print("\nMobile usage (iPhone):")
print("  1. Transfer the q4_k_m .gguf file to the phone (AirDrop / Files app).")
print("  2. Open in an iOS GGUF runner (LLMFarm, PocketPal, Private LLM, etc.).")
print("  3. Use the Qwen ChatML template; set context length <= MAX_SEQ_LENGTH.")

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.8.22: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Forced GGUF EOS to '<|im_end|>' (id=248046); text_config default was 248044 (<|endoftext|>)
Exporting GGUF (single merge -> 3 quants)...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/gguf/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 101067.57it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:48<00:00, 24.34s/it]


Unsloth: Merge process complete. Saved to `/workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m', 'q5_k_m', 'q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: No supported architectures (TEXT or VISION) could be determined from the original script.
[unsloth_zoo.llama_cpp|WARNING]Unsloth: Metadata branding patch target 'self.metadata = gguf.Metadata.load(...)' not found.
[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Qwen3_5ForConditionalGeneration is not supported for MMPROJ conversion. Converting as text-only model.


RuntimeError: Unsloth: GGUF conversion failed: RuntimeError: Unsloth: Failed to convert model to GGUF with command `/usr/bin/python /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile Qwen3.5-4B.BF16.gguf --outtype bf16 --split-max-size 50G /workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/gguf`: Command '['/usr/bin/python', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', 'Qwen3.5-4B.BF16.gguf', '--outtype', 'bf16', '--split-max-size', '50G', '/workspace/training/biblical/output/biblical_qwen3_5_4b_unsloth_4bit_dpo/gguf']' returned non-zero exit status 1.
--- converter stderr ---
Traceback (most recent call last):
  File "/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py", line 18, in <module>
    from conversion import (
ModuleNotFoundError: No module named 'conversion'

## 13. GGUF Export That Works on This Architecture — merged **and** standalone LoRA

Section 12 is unsloth's one-call `save_pretrained_gguf`, and it fails on
`Qwen3_5ForConditionalGeneration`: its patched converter reports no supported TEXT/VISION
architecture and raises `Unsloth: Failed to convert model to GGUF`. This section is the route that
actually produced the shipped SFT GGUFs — merge by hand, then drive llama.cpp's own converters:

1. Merge base + SFT + DPO on **CPU** in bf16 with PEFT into `gguf_build/merged`.
2. Force `<|im_end|>` (248046) as EOS on `config`, `config.text_config` **and** `generation_config`.
   llama.cpp reads the nested `text_config` value on `*ForConditionalGeneration` models, and the
   checkpoint ships `248044` (`<|endoftext|>`) there — leave it and the model never stops.
3. Invert the chat template's thinking default to **off**, matching what SFT and DPO trained on.
   The template is baked into the GGUF and mobile runners cannot pass `enable_thinking`.
4. `convert_hf_to_gguf.py` → BF16, then `llama-quantize` → Q2_K / Q3_K_M / Q4_K_M / Q5_K_M / Q8_0.
5. `convert_lora_to_gguf.py` → a standalone ~82 MB adapter GGUF.
6. Write `Modelfile.merged` and `Modelfile.adapter` for `ollama create`.

**Merged vs. standalone.** The merged quants are self-contained — one file, nothing else to load,
the right choice for phones and for `ollama run`. The standalone adapter is ~82 MB and attaches to a
GGUF of the **unmodified base** (`llama-cli --lora`, Ollama's `ADAPTER`), so one base file can serve
several personas or A/B against the stock model. Applying it on top of the merged file would apply
the same deltas twice. DPO continued the SFT adapter rather than stacking a second one, so there is
exactly one adapter to ship and it carries both stages.

**Verified on this stack:** the LoRA → GGUF step was run against the SFT adapter in this container
(llama.cpp at `/root/.unsloth/llama.cpp`, 256 tensors, 82 MB out). The merge → convert → quantize
half is the same sequence that produced `Qwen3.5-4B-biblical-{BF16,Q2_K,Q3_K_M,Q4_K_M,Q5_K_M,Q8_0}.gguf`.

**Cost:** ~20 GB RAM for the CPU merge, ~31 GB of GGUFs plus an 8.4 GB `gguf_build/merged` staging
copy, and 15–25 minutes. No GPU — safe to run while something else holds the card.

In [ ]:
# =========================== GGUF EXPORT: merged + standalone LoRA ===========================
# Section 12's unsloth exporter does not work on this architecture (see its note). This cell is the
# route that actually produced the shipped SFT GGUFs: merge on CPU with PEFT, then drive
# llama.cpp's own, unpatched converters. Nothing here touches the GPU.
#
# Writes into OUTPUT_BASE_DIR/gguf:
#   <PREFIX>-BF16.gguf                        merged base + SFT + DPO, unquantized   (~7.9 GB)
#   <PREFIX>-Q2_K / Q3_K_M / Q4_K_M / Q5_K_M / Q8_0   quantized merged models
#   <PREFIX>-LoRA-f16.gguf                    standalone adapter, ~82 MB, for
#                                             `llama-cli --lora` or Ollama `ADAPTER`
#   Modelfile.merged, Modelfile.adapter       ready for `ollama create`
#
# The standalone adapter is the SFT+DPO LoRA as one file - DPO continued the SFT adapter rather
# than stacking a second one, so there is exactly one adapter to ship. It must be applied on top of
# a GGUF of the UNMODIFIED base model, never on top of the merged file above.
#
# Cost: ~20 GB RAM during the merge, ~26 GB disk, roughly 15-25 minutes end to end.

import gc, json, os, subprocess, sys, time
from pathlib import Path

import torch, transformers
from transformers import AutoConfig, AutoTokenizer
from peft import PeftModel

GGUF_PREFIX     = "Qwen3.5-4B-biblical-dpo"
GGUF_OUTPUT_DIR = OUTPUT_BASE_DIR / "gguf"
GGUF_MERGED_DIR = OUTPUT_BASE_DIR / "gguf_build" / "merged"
QUANT_TYPES     = ["Q2_K", "Q3_K_M", "Q4_K_M", "Q5_K_M", "Q8_0"]
KEEP_BF16       = True    # set False to delete the 7.9 GB unquantized GGUF once the quants exist

EOT_TOKEN, EOT_ID = "<|im_end|>", 248046

GGUF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_MERGED_DIR.mkdir(parents=True, exist_ok=True)


def _run(cmd, **kw):
    """Run a converter/quantizer and let its output stream into the notebook."""
    print("  $ " + " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True, **kw)


# ---- [0/6] Locate llama.cpp BEFORE spending twenty minutes on a merge ------------------------
def _find_llama_cpp():
    roots = [Path("/root/.unsloth/llama.cpp"), Path("/home/spark/resources/llama.cpp"),
             Path("/workspace/llama.cpp"), Path.home() / "llama.cpp"]
    for root in roots:
        # convert_hf_to_gguf.py, NOT unsloth_convert_hf_to_gguf.py: the patched copy is the one
        # that reports "no supported architectures" and fails on Qwen3_5ForConditionalGeneration.
        conv, conv_lora = root / "convert_hf_to_gguf.py", root / "convert_lora_to_gguf.py"
        quant = next((p for p in (root / "llama-quantize", root / "build" / "bin" / "llama-quantize")
                      if p.exists()), None)
        if conv.exists() and conv_lora.exists() and quant is not None:
            return root, conv, conv_lora, quant
    raise RuntimeError("No usable llama.cpp checkout found. Looked in: "
                       + ", ".join(str(r) for r in roots))


LLAMA_ROOT, CONVERT_HF, CONVERT_LORA, QUANTIZE_BIN = _find_llama_cpp()
_quant_env = os.environ.copy()
_quant_env["LD_LIBRARY_PATH"] = f"{QUANTIZE_BIN.parent}:{_quant_env.get('LD_LIBRARY_PATH', '')}"
print(f"[0/6] llama.cpp: {LLAMA_ROOT}  (quantizer: {QUANTIZE_BIN.name})")

# ---- [1/6] Validate the chat-template patch while failing is still cheap ---------------------
# The stock template pre-closes the think block ONLY when enable_thinking is explicitly false;
# every other path opens one, and this adapter never saw a </think> in training. GGUF bakes the
# template in and mobile runners have no clean way to pass the kwarg, so invert the default here.
THINK_OLD = ("{%- if enable_thinking is defined and enable_thinking is false %}\n"
             "        {{- '<think>\\n\\n</think>\\n\\n' }}\n"
             "    {%- else %}\n"
             "        {{- '<think>\\n' }}\n"
             "    {%- endif %}")
THINK_NEW = ("{%- if enable_thinking is defined and enable_thinking is true %}\n"
             "        {{- '<think>\\n' }}\n"
             "    {%- else %}\n"
             "        {{- '<think>\\n\\n</think>\\n\\n' }}\n"
             "    {%- endif %}")

_tpl_file = LORA_OUTPUT_DIR / "chat_template.jinja"
if _tpl_file.exists():
    _tpl_src = _tpl_file.read_text()
else:
    _tpl_src = json.loads((LORA_OUTPUT_DIR / "tokenizer_config.json").read_text()).get("chat_template")
    if not isinstance(_tpl_src, str):
        raise RuntimeError(f"No chat template found in {LORA_OUTPUT_DIR}")

if _tpl_src.count(THINK_OLD) == 1:
    _tpl_patched = _tpl_src.replace(THINK_OLD, THINK_NEW)
    print("[1/6] chat template: thinking-off default will be applied")
elif THINK_NEW in _tpl_src:
    _tpl_patched = _tpl_src
    print("[1/6] chat template: already defaults to thinking-off")
else:
    raise RuntimeError(f"Chat template patch target not found in {LORA_OUTPUT_DIR} "
                       f"({_tpl_src.count(THINK_OLD)} matches) - inspect it before exporting")

# ---- [2/6] Merge the adapter into the base, on CPU, in bf16 ----------------------------------
for _name in ("model", "tokenizer", "processor", "trainer", "model2", "tokenizer2", "dpo_dataset"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

_arch = AutoConfig.from_pretrained(BASE_LLM, trust_remote_code=True).architectures[0]
print(f"[2/6] loading {BASE_LLM} as {_arch} on CPU in bf16, then merging {LORA_OUTPUT_DIR.name} ...",
      flush=True)
_t0 = time.time()
_merged = getattr(transformers, _arch).from_pretrained(
    BASE_LLM, dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
)
_merged = PeftModel.from_pretrained(_merged, str(LORA_OUTPUT_DIR), device_map="cpu").merge_and_unload()

# ---- EOS: the chat template closes on <|im_end|> (248046) but the checkpoint ships
# text_config.eos_token_id = 248044 (<|endoftext|>). llama.cpp reads the NESTED value on
# *ForConditionalGeneration models, so both levels must be set before conversion, or the model
# never stops and runs on into the next turn header.
_merged.config.eos_token_id = EOT_ID
if getattr(_merged.config, "text_config", None) is not None:
    _merged.config.text_config.eos_token_id = EOT_ID
if getattr(_merged, "generation_config", None) is not None:
    _merged.generation_config.eos_token_id = EOT_ID

_merged.save_pretrained(str(GGUF_MERGED_DIR), safe_serialization=True)
AutoTokenizer.from_pretrained(str(LORA_OUTPUT_DIR)).save_pretrained(str(GGUF_MERGED_DIR))
(GGUF_MERGED_DIR / "chat_template.jinja").write_text(_tpl_patched)

# tokenizer_config.json carries its own embedded copy of the template on this checkpoint.
_tc_path = GGUF_MERGED_DIR / "tokenizer_config.json"
_tc = json.loads(_tc_path.read_text())
_tc["eos_token"] = EOT_TOKEN
if isinstance(_tc.get("chat_template"), str) and THINK_OLD in _tc["chat_template"]:
    _tc["chat_template"] = _tc["chat_template"].replace(THINK_OLD, THINK_NEW)
_tc_path.write_text(json.dumps(_tc, indent=2, ensure_ascii=False))

_cfg_json = json.loads((GGUF_MERGED_DIR / "config.json").read_text())
print(f"      merged in {(time.time() - _t0) / 60:.1f} min -> {GGUF_MERGED_DIR}")
print(f"      eos top-level: {_cfg_json.get('eos_token_id')} | "
      f"text_config: {_cfg_json.get('text_config', {}).get('eos_token_id')} (both must be {EOT_ID})")

del _merged
gc.collect()
torch.cuda.empty_cache()

# ---- [3/6] Merged HF -> BF16 GGUF -------------------------------------------------------------
_bf16_path = GGUF_OUTPUT_DIR / f"{GGUF_PREFIX}-BF16.gguf"
print(f"[3/6] converting merged model to BF16 GGUF ...", flush=True)
_run([sys.executable, CONVERT_HF, GGUF_MERGED_DIR, "--outfile", _bf16_path, "--outtype", "bf16"])

# ---- [4/6] BF16 -> quants (one merge, one conversion, N quantizations) ------------------------
_nthreads = str(os.cpu_count() or 8)
for _q in QUANT_TYPES:
    _out = GGUF_OUTPUT_DIR / f"{GGUF_PREFIX}-{_q}.gguf"
    print(f"[4/6] quantizing -> {_out.name}", flush=True)
    _run([QUANTIZE_BIN, _bf16_path, _out, _q, _nthreads], env=_quant_env)

# ---- [5/6] Standalone LoRA GGUF ---------------------------------------------------------------
# convert_lora_to_gguf.py needs only the BASE model's config, not its weights.
from huggingface_hub import snapshot_download

_base_cfg_dir = snapshot_download(BASE_LLM, allow_patterns=["config.json"])
_lora_gguf = GGUF_OUTPUT_DIR / f"{GGUF_PREFIX}-LoRA-f16.gguf"
print(f"[5/6] converting the adapter itself -> {_lora_gguf.name}", flush=True)
_run([sys.executable, CONVERT_LORA, "--base", _base_cfg_dir, "--outtype", "f16",
      "--outfile", _lora_gguf, LORA_OUTPUT_DIR])

# ---- [6/6] Ollama Modelfiles + summary --------------------------------------------------------
_q_primary = "Q4_K_M" if "Q4_K_M" in QUANT_TYPES else QUANT_TYPES[0]
(GGUF_OUTPUT_DIR / "Modelfile.merged").write_text(
    f"""# ollama create biblical-qwen35-4b-dpo -f Modelfile.merged
FROM ./{GGUF_PREFIX}-{_q_primary}.gguf
PARAMETER stop "{EOT_TOKEN}"
PARAMETER temperature 0.7
"""
)
(GGUF_OUTPUT_DIR / "Modelfile.adapter").write_text(
    f"""# ollama create biblical-qwen35-4b-dpo-lora -f Modelfile.adapter
#
# FROM must point at a GGUF of the UNMODIFIED base model - applying this adapter on top of the
# merged GGUF above would apply the same deltas twice. Build the base GGUF once with:
#   python {CONVERT_HF} <local snapshot of {BASE_LLM}> \\
#       --outfile ./Qwen3.5-4B-base-BF16.gguf --outtype bf16
FROM ./Qwen3.5-4B-base-BF16.gguf
ADAPTER ./{_lora_gguf.name}
PARAMETER stop "{EOT_TOKEN}"
PARAMETER temperature 0.7
"""
)

if not KEEP_BF16:
    _bf16_path.unlink(missing_ok=True)
    print("      removed the unquantized BF16 GGUF (KEEP_BF16=False)")

print(f"\nGGUF export complete: {GGUF_OUTPUT_DIR}")
for _f in sorted(GGUF_OUTPUT_DIR.iterdir()):
    print(f"  {_f.name:52s} {_f.stat().st_size / 1024 / 1024:>9.1f} MB")

print("\nUse it:")
print(f"  llama-cli -m {GGUF_OUTPUT_DIR}/{GGUF_PREFIX}-{_q_primary}.gguf -cnv")
print(f"  llama-cli -m ./Qwen3.5-4B-base-BF16.gguf --lora {_lora_gguf} -cnv")
print(f"  cd {GGUF_OUTPUT_DIR} && ollama create biblical-qwen35-4b-dpo -f Modelfile.merged")
print("\nMobile (iPhone): copy the Q4_K_M file over, open it in LLMFarm / PocketPal / Private LLM,")
print(f"use the Qwen ChatML template, and set context length <= {MAX_SEQ_LENGTH}.")
